# database construction and data cleaning

In [1]:
import duckdb
import pandas as pd
import time

# 1. 初始化本地数据库连接 (持久化存储，下次打开不用再洗数据)
print("正在连接/创建本地 DuckDB 数据库...")
con = duckdb.connect('taobao_project.db')

print("开始执行 Phase 1: 数据清洗与底层宽表构建 (这大概需要 1-3 分钟，请耐心等待)...")
start_time = time.time()

# 2. 核心 SQL：读取、解析时间戳、多维特征提取、过滤边界
# 使用 CREATE TABLE IF NOT EXISTS 避免重复运行报错
con.execute("""
    CREATE TABLE IF NOT EXISTS user_behavior_cleaned AS
    WITH raw_data AS (
        -- 由于原始 CSV 无表头，这里强行指定列名和类型，大幅提升解析速度
        SELECT * FROM read_csv('UserBehavior.csv', 
            header=False,
            columns={
                'user_id': 'BIGINT',
                'item_id': 'BIGINT',
                'category_id': 'BIGINT',
                'behavior_type': 'VARCHAR',
                'timestamp': 'BIGINT'
            }
        )
    )
    SELECT 
        user_id,
        item_id,
        category_id,
        behavior_type,
        -- 将 Unix 时间戳 (秒) 转换为标准的 Timestamp
        to_timestamp(timestamp) AS action_time,
        
        -- 【特征工程】：拆解时间维度，为后续归因分析备足弹药
        CAST(to_timestamp(timestamp) AS DATE) AS action_date,
        EXTRACT(HOUR FROM to_timestamp(timestamp)) AS action_hour,
        DAYOFWEEK(to_timestamp(timestamp)) AS action_weekday 
        -- 注: DuckDB 中 DAYOFWEEK 返回 0(周日) 到 6(周六)
        
    FROM raw_data
    WHERE 
        -- 【异常清洗】：剔除双十一/双十二蓄水期 (11.25-12.03) 之外的脏数据
        CAST(to_timestamp(timestamp) AS DATE) BETWEEN '2017-11-25' AND '2017-12-03';
""")

end_time = time.time()
print(f"✅ 数据清洗完毕！耗时: {round(end_time - start_time, 2)} 秒")

# 3. 结果验证：看看洗出了多少有效数据
print("\n--- 清洗后的底层大盘概况 ---")
summary_df = con.execute("""
    SELECT 
        COUNT(*) AS total_valid_rows, 
        COUNT(DISTINCT user_id) AS total_users,
        MIN(action_date) AS min_date, 
        MAX(action_date) AS max_date 
    FROM user_behavior_cleaned
""").df()

print(summary_df)

# 养成好习惯，执行完关闭连接
con.close()

ModuleNotFoundError: No module named 'pandas'